# 01 Exploration

Load all local CRMLS sold CSV files from `data/raw/` and run initial exploration for the California close price prediction project.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

## Load Data

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

csv_files = sorted(RAW_DATA_DIR.glob("CRMLSSold*.csv"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Found {len(csv_files)} CRMLSSold CSV files")

if not csv_files:
    raise FileNotFoundError(f"No CRMLSSold*.csv files found in {RAW_DATA_DIR}")

for path in csv_files:
    print(path.name)

In [ ]:
def load_crmls_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["source_file"] = path.name
    return df


dataframes = []
file_summary = []

for path in csv_files:
    df = load_crmls_file(path)
    dataframes.append(df)
    file_summary.append(
        {
            "source_file": path.name,
            "rows": len(df),
            "columns": df.shape[1],
        }
    )

raw_df = pd.concat(dataframes, ignore_index=True)
file_summary_df = pd.DataFrame(file_summary)

print(f"Combined shape: {raw_df.shape}")
file_summary_df

In [ ]:
raw_df.head()

In [ ]:
raw_df.info(memory_usage="deep")

## Initial Checks

In [ ]:
key_columns = [
    "ClosePrice",
    "CloseDate",
    "PropertyType",
    "PropertySubType",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "City",
    "CountyOrParish",
    "PostalCode",
]

available_key_columns = [col for col in key_columns if col in raw_df.columns]
missing_key_columns = sorted(set(key_columns) - set(available_key_columns))

print("Available key columns:")
print(available_key_columns)

print("\nMissing expected columns:")
print(missing_key_columns)

In [ ]:
raw_df[available_key_columns].isna().mean().sort_values(ascending=False).to_frame("missing_rate")

In [ ]:
for col in ["PropertyType", "PropertySubType", "MlsStatus"]:
    if col in raw_df.columns:
        display(raw_df[col].value_counts(dropna=False).head(20).to_frame("count"))

## Filter to Project Scope

The task prompt says to model only records where `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.

In [ ]:
project_df = raw_df.copy()

if "CloseDate" in project_df.columns:
    project_df["CloseDate"] = pd.to_datetime(project_df["CloseDate"], errors="coerce")

numeric_columns = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "YearBuilt",
    "Latitude",
    "Longitude",
]

for col in numeric_columns:
    if col in project_df.columns:
        project_df[col] = pd.to_numeric(project_df[col], errors="coerce")

project_df = project_df[
    (project_df["PropertyType"] == "Residential")
    & (project_df["PropertySubType"] == "SingleFamilyResidence")
    & project_df["ClosePrice"].notna()
].copy()

print(f"Raw rows: {len(raw_df):,}")
print(f"Project-scope rows: {len(project_df):,}")
print(f"Date range: {project_df['CloseDate'].min()} to {project_df['CloseDate'].max()}")

project_df[available_key_columns + ["source_file"]].head()

## Basic Exploration

In [ ]:
eda_numeric_columns = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
]

eda_numeric_columns = [col for col in eda_numeric_columns if col in project_df.columns]
project_df[eda_numeric_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
monthly_counts = (
    project_df.dropna(subset=["CloseDate"])
    .assign(close_month=lambda df: df["CloseDate"].dt.to_period("M").astype(str))
    .groupby("close_month")
    .agg(rows=("ListingKey", "size"), median_close_price=("ClosePrice", "median"))
    .reset_index()
)

monthly_counts.tail(24)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(project_df["ClosePrice"].dropna(), bins=80, ax=axes[0])
axes[0].set_title("ClosePrice Distribution")
axes[0].set_xlabel("ClosePrice")

sns.histplot(project_df["ClosePrice"].dropna(), bins=80, log_scale=True, ax=axes[1])
axes[1].set_title("ClosePrice Distribution, Log Scale")
axes[1].set_xlabel("ClosePrice")

plt.tight_layout()

In [ ]:
plot_columns = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt"]
plot_columns = [col for col in plot_columns if col in project_df.columns]

fig, axes = plt.subplots(len(plot_columns), 1, figsize=(12, 4 * len(plot_columns)))

if len(plot_columns) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_columns):
    sns.scatterplot(data=project_df, x=col, y="ClosePrice", alpha=0.2, s=12, ax=ax)
    ax.set_title(f"ClosePrice vs {col}")

plt.tight_layout()

In [ ]:
corr_columns = [col for col in eda_numeric_columns if col in project_df.columns]

plt.figure(figsize=(9, 7))
sns.heatmap(project_df[corr_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlations")
plt.tight_layout()